# Step 2 — MVTec AD dataset exploration
We first inspect the dataset, create a fixed 90/10 train/validation split using only `train/good`, and visualize five bottle test images with their masks.

The official test set stays untouched for final evaluation.

In [ ]:
# Mount Google Drive (where the dataset will be stored)
from google.colab import drive
drive.mount('/content/drive')

# Clone the project repo into this Colab session
%cd /content
!git clone https://github.com/pranitas-cloud/CS-663.git
%cd /content/CS-663


## Dataset download and location
Download **MVTec AD** from the [official page](https://www.mvtec.com/research-teaching/datasets/mvtec-ad), extract it, and place the `mvtec_anomaly_detection` folder in Google Drive. The folder should directly contain `bottle/`, `cable/`, `capsule/`, etc.

In [ ]:
from pathlib import Path
import sys, torch
import matplotlib.pyplot as plt
sys.path.insert(0, '/content/CS-663')
from src.data.dataset import MVTecDataset, available_categories

# EDIT THIS path to the folder that directly contains bottle/, cable/, etc.
DATA_ROOT = Path('/content/drive/MyDrive/mvtec_anomaly_detection')
print('Root exists:', DATA_ROOT.exists())
print('Categories:', available_categories(DATA_ROOT))
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU detected')

In [ ]:
# Pilot category: bottle
train_ds = MVTecDataset(DATA_ROOT, 'bottle', 'train', val_fraction=0.10, seed=42)
val_ds = MVTecDataset(DATA_ROOT, 'bottle', 'val', val_fraction=0.10, seed=42)
test_ds = MVTecDataset(DATA_ROOT, 'bottle', 'test')
print(f'train-good: {len(train_ds)} | val-good: {len(val_ds)} | test: {len(test_ds)}')
print('Train/val overlap:', bool(set(s[0] for s in train_ds.samples) & set(s[0] for s in val_ds.samples)))
print('Test defect types:', sorted(set(s[2] for s in test_ds.samples)))

## Visualize five test images and ground-truth masks
Normal test images have empty masks. White pixels in an anomalous sample's mask indicate the annotated defect region.

In [ ]:
good_ids = [i for i, s in enumerate(test_ds.samples) if s[2] == 'good']
bad_ids = [i for i, s in enumerate(test_ds.samples) if s[2] != 'good']
chosen = (good_ids[:2] + bad_ids[:3])[:5]
if len(chosen) < 5: chosen = list(range(min(5, len(test_ds))))

fig, axes = plt.subplots(len(chosen), 2, figsize=(8, 3.2 * len(chosen)))
if len(chosen) == 1: axes = axes[None, :]
for row, idx in enumerate(chosen):
    sample = test_ds[idx]
    axes[row, 0].imshow(sample['image'].permute(1, 2, 0).numpy())
    axes[row, 0].set_title(f"Image — {sample['defect_type']}")
    axes[row, 1].imshow(sample['mask'].squeeze(0).numpy(), cmap='gray', vmin=0, vmax=1)
    axes[row, 1].set_title('Ground-truth mask')
    for ax in axes[row]: ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Basic loader sanity checks
sample = test_ds[0]
assert sample['image'].shape == (3, 256, 256)
assert sample['mask'].shape == (1, 256, 256)
assert sample['image'].min() >= 0 and sample['image'].max() <= 1
assert set(torch.unique(sample['mask']).tolist()).issubset({0.0, 1.0})
print('Loader sanity checks passed.')